# One-Dimensional Maps, Fixed Points, and Root-Finding
### MATH-3515 — Numerical Analysis — Week 4

This notebook follows the Week 3 handout: one-dimensional maps and orbits, fixed
points and stability, cobweb diagrams, and the bisection method. It also adds
an implementation of **Newton's method** and explores how quickly each
root-finding method converges.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## 2. One-Dimensional Maps & Orbits

Recall: given $x_{n+1} = g(x_n)$ and an initial condition $x_0$, the sequence
$x_0, x_1, x_2, \dots$ is called the **orbit** of $x_0$ under $g$.

In [ ]:
def orbit(g, x0, n):
    """Compute the first n+1 terms of the orbit x_0, x_1, ..., x_n under x_{k+1} = g(x_k)."""
    xs = np.empty(n + 1)
    xs[0] = x0
    for i in range(n):
        xs[i + 1] = g(xs[i])
    return xs

# Demo: g(x) = cos(x)
xs = orbit(np.cos, x0=1.0, n=30)
print("Last 5 terms of the orbit:", xs[-5:])
print("Looks like it's converging to a fixed point near", xs[-1])

## 3. Linear Maps

For $x_{n+1} = \alpha x_n$, the explicit form is $x_n = \alpha^n x_0$. Behavior depends
on $|\alpha|$:

- $|\alpha| < 1 \Rightarrow x_n \to 0$
- $|\alpha| = 1 \Rightarrow$ constant or alternating between $\pm x_0$
- $|\alpha| > 1 \Rightarrow$ divergence

Let's verify the closed form numerically and visualize each case.

In [ ]:
def linear_map_orbit(alpha, x0, n):
    return orbit(lambda x: alpha * x, x0, n)

# Verify against closed form x_n = alpha^n * x0
for alpha in [0.5, 1.0, -1.0, 1.5]:
    xs = linear_map_orbit(alpha, x0=1.0, n=10)
    xn_closed = alpha ** np.arange(11) * 1.0
    assert np.allclose(xs, xn_closed)
print("Iterative computation matches closed form x_n = alpha**n * x0 in all cases.")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
n = 15
for alpha, label in [(0.7, r'$\alpha=0.7$ ($|\alpha|<1$)'),
                      (1.0, r'$\alpha=1$ ($|\alpha|=1$)'),
                      (-1.0, r'$\alpha=-1$ ($|\alpha|=1$)'),
                      (1.2, r'$\alpha=1.2$ ($|\alpha|>1$)')]:
    xs = linear_map_orbit(alpha, x0=1.0, n=n)
    ax.plot(range(n + 1), xs, 'o-', label=label)

ax.axhline(0, color='gray', lw=0.5)
ax.set_xlabel('$n$')
ax.set_ylabel('$x_n$')
ax.set_title('Linear map orbits for different $\\alpha$')
ax.legend()
plt.show()

## 4. Fixed Points & Stability

A point $\bar{x}$ is a **fixed point** of $g$ if $g(\bar{x}) = \bar{x}$.
Linearizing around $\bar{x}$ gives $\theta_{n+1} \approx g'(\bar{x})\,\theta_n$, so
with $\lambda = g'(\bar{x})$:

- $|\lambda| < 1 \Rightarrow$ **stable** fixed point
- $|\lambda| > 1 \Rightarrow$ **unstable** fixed point

We estimate $g'(\bar{x})$ with a central finite difference and classify each
fixed point from the handout's examples.

In [ ]:
def numerical_derivative(g, x, h=1e-6):
    return (g(x + h) - g(x - h)) / (2 * h)

def classify_fixed_point(g, xbar, h=1e-6):
    lam = numerical_derivative(g, xbar, h)
    if abs(lam) < 1:
        status = "stable"
    elif abs(lam) > 1:
        status = "unstable"
    else:
        status = "marginal (|lambda| = 1)"
    return lam, status

In [ ]:
examples = {
    "cos(x)":          (np.cos,                     0.7390851332151607),
    "x^3":              (lambda x: x**3,             0.0),
    "(x + 2/x)/2":      (lambda x: 0.5*(x + 2/x),    np.sqrt(2)),
}

for name, (g, xbar) in examples.items():
    lam, status = classify_fixed_point(g, xbar)
    print(f"{name:15s}  xbar = {xbar:.5f}   g'(xbar) = {lam:8.5f}   -> {status}")

The logistic map $x_{n+1} = r\,x_n(1 - x_n)$ has a nonzero fixed point at
$\bar{x} = 1 - 1/r$. Its stability depends on $r$ — this is the map's classic
route to period-doubling and chaos as $r$ increases.

In [ ]:
def logistic(r):
    return lambda x: r * x * (1 - x)

for r in [2.5, 3.2, 3.5]:
    xbar = 1 - 1/r
    lam, status = classify_fixed_point(logistic(r), xbar)
    print(f"logistic, r={r}:  xbar = {xbar:.5f}   g'(xbar) = {lam:8.5f}   -> {status}")

## 5. Cobweb Diagrams

A cobweb diagram traces $x_0 \to x_1 \to x_2 \to \dots$ graphically: from
$x_n$ on the axis, move vertically to the curve $y=g(x)$ to get $x_{n+1}$,
then horizontally to the line $y = x$, then vertically again — repeat.

In [ ]:
def cobweb(g, x0, n, xlim, ax=None, title=None):
    """Draw a cobweb diagram for x_{k+1} = g(x_k), starting at x0, for n steps."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))

    t = np.linspace(xlim[0], xlim[1], 400)
    ax.plot(t, g(t), 'b-', label='$y=g(x)$')
    ax.plot(t, t, 'k--', label='$y=x$')

    x = x0
    for _ in range(n):
        y = g(x)
        ax.plot([x, x], [x, y], 'r-', lw=0.8)   # vertical: to the curve
        ax.plot([x, y], [y, y], 'r-', lw=0.8)   # horizontal: to y=x
        x = y

    ax.set_xlim(xlim)
    ax.set_ylim(xlim)
    ax.set_xlabel('$x_n$')
    ax.set_ylabel('$x_{n+1}$')
    ax.set_aspect('equal')
    if title:
        ax.set_title(title)
    ax.legend(fontsize=8)
    return ax

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
cobweb(np.cos, x0=1.0, n=25, xlim=(0, 1.5), ax=axes[0], title='(a) $x_{n+1}=\\cos(x_n)$')
cobweb(lambda x: 0.5*(x + 2/x), x0=1.0, n=6, xlim=(0.5, 2), ax=axes[1],
       title=r'(c) $x_{n+1}=\frac{1}{2}(x_n+2/x_n)$, $\bar{x}=\sqrt{2}$')
cobweb(logistic(2.5), x0=0.1, n=25, xlim=(0, 1), ax=axes[2], title='logistic, $r=2.5$ (stable)')
plt.tight_layout()
plt.show()

All three spiral or step inward toward their fixed points — consistent with
the "stable" classification found above. Try `logistic(3.5)` in the cell
below: since $|g'(\bar{x})| > 1$ there, the cobweb should move *away* from
the fixed point instead of converging to it.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
cobweb(logistic(3.5), x0=0.72, n=40, xlim=(0, 1), ax=ax, title='logistic, $r=3.5$ (unstable fixed point)')
plt.show()

## 6. Bisection Method

**Theorem.** If $f$ is continuous on $[a,b]$ and $f(a)f(b) < 0$, then $f$ has
a root in $(a,b)$.

**Algorithm.** While $\frac{b-a}{2} > \text{tol}$: let $c = \frac{a+b}{2}$;
if $f(a)f(c) < 0$ set $b = c$, else set $a = c$.

In [ ]:
def bisection(f, a, b, tol=1e-8, max_iter=100):
    """Find a root of f in [a, b] via bisection. Returns (root, history)."""
    assert f(a) * f(b) < 0, "f(a) and f(b) must have opposite signs"
    history = []
    c = (a + b) / 2
    for i in range(max_iter):
        c = (a + b) / 2
        fc = f(c)
        history.append({'step': i, 'a': a, 'b': b, 'c': c, 'f(c)': fc, 'width': (b - a) / 2})
        if fc == 0 or (b - a) / 2 < tol:
            break
        if f(a) * fc < 0:
            b = c
        else:
            a = c
    return c, history

In [ ]:
f = lambda x: x**3 + x - 1

root_b, hist_b = bisection(f, 0, 1, tol=1e-8)

print(f"Bisection root estimate: {root_b:.10f}")
print(f"Number of iterations:    {len(hist_b)}\n")

print(f"{'Step':>4} {'a':>10} {'b':>10} {'c':>12} {'f(c)':>14}")
for row in hist_b[:6]:
    print(f"{row['step']:>4} {row['a']:>10.6f} {row['b']:>10.6f} {row['c']:>12.8f} {row['f(c)']:>14.8f}")
print("...")

This reproduces the table worked through in lecture (steps 0–3 shown above match the handout).

## 7. Newton's Method

Newton's method uses the tangent line at $x_n$ to get a (usually) much better
next guess:

$$x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}$$

Unlike bisection, it needs $f'$ and a single starting point $x_0$ rather than
a bracketing interval — and it isn't guaranteed to converge, but when it does,
it's typically much faster.

In [ ]:
def newton(f, fprime, x0, tol=1e-12, max_iter=100):
    """Find a root of f near x0 via Newton's method. Returns (root, history)."""
    history = []
    x = x0
    for i in range(max_iter):
        fx = f(x)
        history.append({'step': i, 'x': x, 'f(x)': fx})
        if abs(fx) < tol:
            break
        x = x - fx / fprime(x)
    return x, history

In [ ]:
fprime = lambda x: 3*x**2 + 1  # derivative of f(x) = x^3 + x - 1

root_n, hist_n = newton(f, fprime, x0=1.0, tol=1e-12)

print(f"Newton root estimate:  {root_n:.12f}")
print(f"Number of iterations:  {len(hist_n)}\n")

print(f"{'Step':>4} {'x':>18} {'f(x)':>16}")
for row in hist_n:
    print(f"{row['step']:>4} {row['x']:>18.12f} {row['f(x)']:>16.3e}")

Newton reaches machine precision in about 5–6 steps, versus roughly 27 for bisection at a much looser tolerance — a first hint that it converges much faster. Let's quantify that.

## 8. Convergence Rate: Bisection vs. Newton

**Theory recap:**
- Bisection: the interval half-width exactly halves each step, so the error
  bound decays like $2^{-n}$ — **linear convergence**, order $p=1$.
- Newton: near a simple root, the error roughly *squares* each step
  ($e_{n+1} \approx C\, e_n^2$) — **quadratic convergence**, order $p=2$.

We'll check both claims empirically, using a high-accuracy Newton root as
ground truth.

In [ ]:
true_root = root_n  # accurate to ~1e-15

errors_b = np.array([abs(row['c'] - true_root) for row in hist_b])
errors_n = np.array([abs(row['x'] - true_root) for row in hist_n])

fig, ax = plt.subplots(figsize=(7, 5))
ax.semilogy(range(len(errors_b)), errors_b, 'o-', label='Bisection')
ax.semilogy(range(len(errors_n)), errors_n, 's-', label="Newton's method")
ax.set_xlabel('Iteration $n$')
ax.set_ylabel(r'Error $|x_n - \bar{x}|$ (log scale)')
ax.set_title('Convergence comparison')
ax.legend()
plt.show()

On the log scale, bisection's error decreases in a straight-ish line
(linear convergence), while Newton's error plunges almost vertically after a
couple of steps — the hallmark of quadratic convergence: once you're close,
the number of correct digits roughly **doubles** each iteration.

### Estimating the empirical convergence order

For a sequence converging with order $p$, consecutive errors satisfy
$e_{n+1} \approx C\,e_n^p$, so

$$p \approx \frac{\log(e_{n+1}/e_n)}{\log(e_n/e_{n-1})}$$

For **bisection**, we use the interval half-width $\frac{b-a}{2}$ instead of
the raw error — the raw error can bounce around within the interval (the
root isn't always near the midpoint), while the half-width shrinks by
*exactly* a factor of 2 every step, giving a clean order estimate.

In [ ]:
def empirical_order(errors):
    errors = np.asarray(errors)
    errors = errors[errors > 0]
    return np.log(errors[2:] / errors[1:-1]) / np.log(errors[1:-1] / errors[:-2])

widths_b = np.array([row['width'] for row in hist_b])
p_bisection = empirical_order(widths_b)
p_newton = empirical_order(errors_n)

print("Bisection empirical order (using interval half-width):")
print(np.round(p_bisection[:8], 3), " ... -> approaches 1 (linear)\n")

print("Newton empirical order (using true error):")
print(np.round(p_newton, 3), " -> approaches 2 (quadratic)")

As expected: bisection's order estimate sits right at **1**, while
Newton's climbs toward **2** as the iterate gets close to the root (the
early steps are less accurate because the local linearization $f \approx
f(x_n) + f'(x_n)(x-x_n)$ isn't yet a great approximation far from
$\bar{x}$).

### Newton's method isn't always well-behaved

Newton's method assumes $x_0$ is "close enough" to the root and that
$f'(x_n) \neq 0$ along the way. With a poor starting guess it can oscillate,
diverge, or fail outright. Here's a small example: $f(x) = x^3 - 2x + 2$
starting from $x_0 = 0$, where the iteration falls into a 2-cycle and never
converges.

In [ ]:
f2 = lambda x: x**3 - 2*x + 2
f2prime = lambda x: 3*x**2 - 2

root_bad, hist_bad = newton(f2, f2prime, x0=0.0, max_iter=10)

print(f"{'Step':>4} {'x':>10}")
for row in hist_bad:
    print(f"{row['step']:>4} {row['x']:>10.4f}")
print("\n-> x oscillates between 0 and 1 forever; Newton never converges from this x0.")
print("   Bisection, by contrast, is guaranteed to converge whenever f(a)f(b) < 0.")

## 9. Summary

| | Bisection | Newton's Method |
|---|---|---|
| Needs | $f$, bracket $[a,b]$ with $f(a)f(b)<0$ | $f$, $f'$, single guess $x_0$ |
| Guaranteed to converge? | Yes, if bracket is valid | No — depends on $x_0$ |
| Convergence order | Linear ($p=1$) | Quadratic ($p=2$), near a simple root |
| Typical cost per step | 1 function evaluation | 1 function + 1 derivative evaluation |

**Takeaways:**
- Fixed-point stability is governed entirely by $|g'(\bar{x})|$: less than 1
  is stable, greater than 1 is unstable — this is exactly what the cobweb
  diagrams show visually.
- Bisection trades speed for a strong guarantee: it always converges once you
  have a valid bracket.
- Newton trades that guarantee for speed: near a well-behaved root it
  converges dramatically faster, but it can fail without a decent starting
  guess.
- In practice, many solvers *combine* the two: bracket with bisection first,
  then switch to Newton (or a secant-type method) once close to the root.